# Sequential Fine Tuning in Google Colab
This notebook mounts Google Drive, sets up the workspace, and runs the sequential fine tuning Optuna study. Files will be saved directly to your Google Drive.

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# Adjust this path to where your project is located in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/end_of_degree_project'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f"Changed working directory to {os.getcwd()}")
else:
    print(f"Directory {PROJECT_DIR} not found. Please create it or change the path.")

In [ ]:
!pip install optuna torch torchvision pandas scikit-learn tqdm

In [ ]:
import torch, optuna
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import HuberLoss, L1Loss, MSELoss
from sklearn.model_selection import train_test_split
from torchvision.models import swin_v2_s, Swin_V2_S_Weights
import gc

# Imports from src (Make sure you are in the project directory)
from src.dataset import FoodDataset
from src.helpers.models import freeze, unfreeze
from src.helpers.ml import standardize, train_eval_loop

torch.cuda.empty_cache() if torch.cuda.is_available() else print('NO CUDA 🙉')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT = 'img_path'
TARGETS = ['fat_g', 'carb_g', 'prot_g']
SEED = 1
BS = 128

In [ ]:
# Make sure food_dataset.csv is available in the data folder within your project
df = pd.read_csv('data/food_dataset.csv')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df[TARGETS], test_df[TARGETS] = standard
ize(train_df[TARGETS], test_df[TARGETS])
val_df, hidden_df = train_test_split(test_df, test_size=0.5, random_state=SEED)

In [ ]:
def get_swin_v2_s_pretrained():
    weights = Swin_V2_S_Weights.DEFAULT
    transforms = weights.transforms()
    model = swin_v2_s(weights=weights)
    return model, transforms

def objective(trial):
    N_UNITS = trial.suggest_int('n_units', 256, 1024)
    DROPOUT = trial.suggest_float('dropout', 0.1, 0.5)
    FE_LR = trial.suggest_float('fe_lr', 1e-5, 1e-2, log=True)
    FT_LR = trial.suggest_float('ft_lr', 1e-6, 1e-4, log=True)
    FE_WEIGHT_DECAY = trial.suggest_float('fe_weight_decay', 1e-4, 1e-1, log=True)
    FT_WEIGHT_DECAY = trial.suggest_float('ft_weight_decay', 1e-4, 1e-1, log=True)
    FE_EPOCHS = trial.suggest_int('fe_epochs', 5, 20)
    FT_EPOCHS = trial.suggest_int('ft_epochs', 10, 50)
    LOSS = trial.suggest_categorical('loss', ['L1', 'MSE', 'Huber'])

    model, transforms = get_swin_v2_s_pretrained()
    freeze(model)
    model.head = nn.Sequential(
        nn.Linear(model.head.in_features, N_UNITS),
        nn.ReLU(),
        nn.Dropout(DROPOUT),
        nn.Linear(N_UNITS, 3)
    )
    model = model.to(device)

    train_ds = FoodDataset(train_df, transform=transforms, input=INPUT, targets=TARGETS)
    val_ds = FoodDataset(val_df, transform=transforms, input=INPUT, targets=TARGETS)
    
    # Num workers reduced for Colab to prevent shared memory issues/bottlenecks
    train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)

    try:
        criterions = { 'L1': L1Loss(), 'MSE': MSELoss(), 'Huber': HuberLoss() }

        train_eval_loop(
            model = model,
            epochs = FE_EPOCHS,
            train_loader = train_loader,
            test_loader = val_loader,
            criterion = criterions[LOSS],
            device = device,
            trial = trial,
            starting_epoch = 0,
            optimizer = torch.optim.AdamW(model.head.parameters(), lr=FE_LR, weight_decay=FE_WEIGHT_DECAY),
        )

        unfreeze(model)

        ft_results = train_eval_loop(
            model = model,
            epochs = FT_EPOCHS,
            train_loader = train_loader,
            test_loader = val_loader,
            criterion = criterions[LOSS],
            device = device,
            trial = trial,
            starting_epoch = FE_EPOCHS,
            optimizer = torch.optim.AdamW(
                [
                    {'params': model.features.parameters(), 'lr': FT_LR}, 
                    {'params': model.head.parameters(), 'lr': FE_LR}
                ], 
                weight_decay=FT_WEIGHT_DECAY
            )
        )

        last_epoch_avg_loss = ft_results['val'][-1][-1]
        return last_epoch_avg_loss

    finally:
        # clean up to avoid linux killing process due to OOM 
        if 'model' in locals(): del model
        if 'train_ds' in locals(): del train_ds
        if 'test_ds' in locals(): del test_ds
        if 'val_ds' in locals(): del val_ds
        if 'train_loader' in locals(): del train_loader
        if 'test_loader' in locals(): del test_loader
        if 'val_loader' in locals(): del val_loader
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# Create a path directly in Google Drive to store the results persistently
google_drive_path = '/content/drive/MyDrive/end_of_degree_project/data/studies'
Path(google_drive_path).mkdir(exist_ok=True, parents=True)

study_name = 'sequential_fine_tuning'
db_path = f'{google_drive_path}/{study_name}.db'
print(f"Saving Optuna study to: {db_path}")

study = optuna.create_study(
    study_name=study_name, 
    storage=f'sqlite:///{db_path}',
    direction='minimize',
    load_if_exists=True,
    pruner=optuna.pruners.HyperbandPruner()
)
study.optimize(objective, n_trials=500)